# Notebook 1 — Ingestão e Verificação dos Dados

Este notebook gera os datasets de exemplo e os carrega no HDFS.

In [ ]:
import subprocess
import os

# Gerar os dados
print('Gerando datasets...')
result = subprocess.run(['python3', '/notebooks/../scripts/generate_data.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERRO:', result.stderr)

In [ ]:
import subprocess

def hdfs(cmd):
    result = subprocess.run(f'hdfs dfs {cmd}', shell=True,
                            capture_output=True, text=True)
    return result.stdout + result.stderr

# Criar diretórios no HDFS
print(hdfs('-mkdir -p /data/csv /data/json /data/parquet'))

# Enviar arquivos
for fmt, src, dst in [
    ('CSV',     '/data/vendas.csv',     '/data/csv/vendas.csv'),
    ('JSON',    '/data/vendas.json',    '/data/json/vendas.json'),
    ('Parquet', '/data/vendas.parquet', '/data/parquet/vendas.parquet'),
]:
    print(f'Enviando {fmt}...')
    print(hdfs(f'-put -f {src} {dst}'))

print('Ingestão concluída!')

In [ ]:
# Verificar arquivos no HDFS
print('=== Estrutura HDFS ===')
print(hdfs('-ls -R /data'))
print()
print('=== Status do cluster ===')
result = subprocess.run('hdfs dfsadmin -report', shell=True,
                        capture_output=True, text=True)
print(result.stdout[:2000])

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Verificacao_Ingestao') \
    .master('spark://namenode:7077') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

# Leitura dos 3 formatos
df_csv     = spark.read.option('header', 'true').option('inferSchema', 'true') \
                  .csv('hdfs://namenode:9000/data/csv/vendas.csv')
df_json    = spark.read.json('hdfs://namenode:9000/data/json/vendas.json')
df_parquet = spark.read.parquet('hdfs://namenode:9000/data/parquet/vendas.parquet')

print(f'CSV     → {df_csv.count():,} linhas | Schema: {df_csv.columns}')
print(f'JSON    → {df_json.count():,} linhas | Schema: {df_json.columns}')
print(f'Parquet → {df_parquet.count():,} linhas | Schema: {df_parquet.columns}')

df_csv.show(5)

In [ ]:
spark.stop()
print('Sessão Spark encerrada.')